用于提取向量的代码

In [1]:
import os
import json
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.decomposition import PCA
from tqdm import tqdm

MODEL_PATH = r"E:\Ajou作业\AI Reserch\REPE复现\model\Qwen2.5-0.5B-Instruct"
JSON_PATH = r"E:\Ajou作业\AI Reserch\REPE复现\data\morality_templated_pairs.json"

LAYERS_TO_EXTRACT = [14, 16, 18, 20]

BATCH_SIZE = 8       # 严格控制批大小，契合 8GB 显存容量
MAX_LENGTH = 512     # 严格硬截断，彻底阻断长文本导致的显存暴涨
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"配置就绪 | 目标设备: {DEVICE} | 预设截断长度: {MAX_LENGTH}")

配置就绪 | 目标设备: cuda | 预设截断长度: 512


In [5]:
def load_templated_data(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"未找到目标 JSON 数据，请确保数据清洗阶段成功: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_last_token_hidden_states(model, tokenizer, prompts, layer_idx):
    """
    自回归模型核心对齐提取函数
    采用 Causal 左 Padding 机制，精准切片 Residual Stream 中聚合了整句语义的最后一个有效 Token 隐状态
    """
    tokenizer.padding_side = "left"  # 强制左 Padding 保证右侧 Token 物理对齐
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # 获取目标层张量 shape: (batch_size, seq_len, hidden_dim)
    hidden_states = outputs.hidden_states[layer_idx]

    # 利用 attention_mask 计算消除 padding 后的真实最后一维 Token 索引
    attention_mask = inputs["attention_mask"]
    last_token_indices = attention_mask.sum(dim=1) - 1

    batch_indices = torch.arange(len(prompts), device=DEVICE)
    last_token_states = hidden_states[batch_indices, last_token_indices, :]

    # 及时解包并释放 GPU 显存中的非必要大张量（防止显存随循环累积）
    del outputs

    return last_token_states.cpu().to(torch.float32).numpy()

In [6]:
# 载入常识道德数据并做基础格式验证
cm_pairs = load_templated_data(JSON_PATH)
print(f"成功加载常识道德对比对，总样本数: {len(cm_pairs)}")
print("\n[数据结构验证] 提取第 0 组样本正向 Prompt 开头:")
print(cm_pairs[0]["high"][:120] + " ...")

成功加载常识道德对比对，总样本数: 6354

[数据结构验证] 提取第 0 组样本正向 Prompt 开头:
USER: Consider the acceptability of the action in the following scenario:
Scenario: I secured the loan because I would m ...


In [7]:
print(f"正在从本地路径读取 Tokenizer 与 Model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",  # 激活内核级闪光注意力机制，极致压控显存峰值
    local_files_only=True
)
model.eval()
print("-> 离线底座模型加载成功，显存池初始化完毕，可安全执行高并发推理。")

正在从本地路径读取 Tokenizer 与 Model...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


-> 离线底座模型加载成功，显存池初始化完毕，可安全执行高并发推理。


In [8]:
output_dir = os.path.dirname(JSON_PATH)

# 开始遍历 14, 16, 18, 20 四个核心层
for target_layer in LAYERS_TO_EXTRACT:
    print(f"\n" + "="*25 + f" 启动第 {target_layer} 层向量提取 " + "="*25)
    high_states = []
    low_states = []

    # 分批次流式前向传播，捕获高维激活值
    for i in tqdm(range(0, len(cm_pairs), BATCH_SIZE), desc=f"Layer {target_layer} Forward"):
        batch = cm_pairs[i : i + BATCH_SIZE]
        batch_high = [item["high"] for item in batch]
        batch_low = [item["low"] for item in batch]

        high_act = get_last_token_hidden_states(model, tokenizer, batch_high, target_layer)
        low_act = get_last_token_hidden_states(model, tokenizer, batch_low, target_layer)

        high_states.append(high_act)
        low_states.append(low_act)

    # 垂直级联构成特征全量矩阵
    H_high = np.vstack(high_states)
    H_low = np.vstack(low_states)

    # 数学解耦：无监督几何空间差值计算
    diff_vectors = H_high - H_low
    # 行级 L2 归一化，剔除句子间绝对长度的模长干扰
    norms = np.linalg.norm(diff_vectors, axis=1, keepdims=True)
    normalized_diff = diff_vectors / (norms + 1e-8)

    # 执行主成分分析 (PCA) 抽取代表道德特征的最大方差轴
    pca = PCA(n_components=1)
    pca.fit(normalized_diff)
    reading_vector = pca.components_[0]

    explained_variance = pca.explained_variance_ratio_[0]
    print(f"-> 第 {target_layer} 层 PCA 拟合完毕。PC1 方差解释比: {explained_variance:.4f}")

    # 超空间基底正负号投影校准 (Sign Alignment)
    high_scores = np.dot(H_high, reading_vector)
    low_scores = np.dot(H_low, reading_vector)

    if np.mean(high_scores) < np.mean(low_scores):
        print(f"   [校准] 检测到第 {target_layer} 层超空间基底反转，执行 [-1] 符号矫正。")
        reading_vector = -reading_vector

    # 固化二进制存储到本地
    vector_save_path = os.path.join(output_dir, f"cm_reading_vector_layer{target_layer}.npy")
    np.save(vector_save_path, reading_vector)
    print(f"   [成功] 第 {target_layer} 层道德控制向量已保存至本地。")

print("\n" + "#"*20 + " 【全线复现第一阶段成功】多层道德控制向量全部固化完毕！ " + "#"*20)


========================= 启动第 14 层向量提取 =========================


Layer 14 Forward: 100%|██████████| 795/795 [05:43<00:00,  2.31it/s]


-> 第 14 层 PCA 拟合完毕。PC1 方差解释比: 0.1772
   [成功] 第 14 层道德控制向量已保存至本地。

========================= 启动第 16 层向量提取 =========================


Layer 16 Forward: 100%|██████████| 795/795 [05:42<00:00,  2.32it/s]


-> 第 16 层 PCA 拟合完毕。PC1 方差解释比: 0.1976
   [成功] 第 16 层道德控制向量已保存至本地。

========================= 启动第 18 层向量提取 =========================


Layer 18 Forward: 100%|██████████| 795/795 [05:40<00:00,  2.33it/s]


-> 第 18 层 PCA 拟合完毕。PC1 方差解释比: 0.1925
   [成功] 第 18 层道德控制向量已保存至本地。

========================= 启动第 20 层向量提取 =========================


Layer 20 Forward: 100%|██████████| 795/795 [05:45<00:00,  2.30it/s]


-> 第 20 层 PCA 拟合完毕。PC1 方差解释比: 0.1684
   [成功] 第 20 层道德控制向量已保存至本地。

#################### 【全线复现第一阶段成功】多层道德控制向量全部固化完毕！ ####################


本质上是一个 896 维的高维单位方向向量。它在 Qwen2.5-0.5B 第 X 层的激活空间（Activation Space）中，精准指向了“常识道德/行为可接受度”这一宏观语义维度的正方向